# 4장 과제: ISL Default 데이터 로지스틱 회귀 분석
**과목**: 인공지능기반 데이터분석 | **제출자**: 정장영

---
**분석 목표**: Default 데이터를 이용하여 로지스틱 회귀 모형을 적합하고,
오즈비 유도 / 변수 선택 / 모형 해석 / 성능 평가 / ROC·PR 곡선을 분석한다.


## 환경 설정 및 데이터 로드

In [ ]:
# ============================================================
# 4장 과제: ISL Default 데이터로 로지스틱 회귀 분석
# 과목: 인공지능기반 데이터분석
# ============================================================

# ── 패키지 설치 ──────────────────────────────────────────────
!pip install ISLP -q

# ── 패키지 로드 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from scipy import stats
from ISLP import load_data

# ── 한글 폰트 설정 (Colab 환경) ────────────────────────────
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
try:
    import japanize_matplotlib
except ImportError:
    pass

# ── 데이터 로드 ──────────────────────────────────────────────
Default = load_data('Default')
print("=" * 60)
print("📊 Default 데이터셋 기본 정보")
print("=" * 60)
print(f"행(관측값) 수: {Default.shape[0]:,}개")
print(f"열(변수)  수: {Default.shape[1]}개")
print()
print("변수 설명:")
print("  default : 채무불이행 여부 (Yes=불이행, No=정상)")
print("  student : 학생 여부 (Yes=학생, No=비학생)")
print("  balance : 신용카드 잔액 (단위: 달러)")
print("  income  : 연간 소득 (단위: 달러)")
print()
print(Default.head(10))
print()
print("채무불이행 비율:")
print(Default['default'].value_counts(normalize=True).round(4) * 100, "%")


## 문제 1. 오즈비(Odds Ratio) 유도

In [ ]:
# ============================================================
# 문제 1. 오즈비(Odds Ratio) 유도
# ============================================================
print("=" * 60)
print("📐 문제 1. 오즈비 유도")
print("=" * 60)
print("""
【통계 용어 정리】
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ 로지스틱 회귀 (Logistic Regression)
  - 결과가 0 또는 1인 분류 문제에 사용하는 회귀 분석
  - 어떤 사건이 일어날 확률(p)을 예측함

▶ 오즈 (Odds)
  - 사건이 일어날 확률 / 일어나지 않을 확률
  - Odds = p / (1-p)
  - 예) p=0.8이면 Odds = 0.8/0.2 = 4  →  "이길 확률이 질 확률의 4배"

▶ 로그 오즈 / 로짓 (log-odds / logit)
  - 오즈에 로그를 취한 값
  - logit(p) = log(p/(1-p))  ← 로지스틱 회귀의 좌변

▶ 오즈비 (Odds Ratio)
  - 두 상황에서 오즈의 비율
  - X가 1 증가할 때 오즈가 몇 배 변하는지 측정
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

【모형 정의】
  log(p / (1-p)) = β₀ + β₁ × balance

  양변에 exp()를 취하면:
  p / (1-p) = exp(β₀ + β₁ × balance) = exp(β₀) × exp(β₁)^balance
""")

import sympy as sp

b0, b1 = sp.symbols('beta_0 beta_1')

print("(1) balance = 0 일 때의 오즈:")
odds_0 = sp.exp(b0 + b1 * 0)
print(f"   Odds(balance=0) = exp(β₀ + β₁×0) = exp(β₀)")
print(f"   → {odds_0}")
print()

print("(2) balance = 1 일 때의 오즈:")
odds_1 = sp.exp(b0 + b1 * 1)
print(f"   Odds(balance=1) = exp(β₀ + β₁×1) = exp(β₀) × exp(β₁)")
print(f"   → {odds_1}")
print()

print("(3) Odds Ratio (balance가 1 증가할 때):")
OR = sp.simplify(odds_1 / odds_0)
print(f"   OR = Odds(balance=1) / Odds(balance=0)")
print(f"      = [exp(β₀) × exp(β₁)] / exp(β₀)")
print(f"      = exp(β₁)")
print(f"   → {OR}")
print()
print("   ✅ 의미: balance가 1 단위 증가할 때, 채무불이행 오즈는 exp(β₁)배가 됨")
print()

print("(4) β₁ = 0.005499 일 때 오즈비 계산:")
beta1_val = 0.005499
OR_val = np.exp(beta1_val)
print(f"   OR = exp(0.005499) = {OR_val:.6f}")
print(f"   ✅ 해석: balance가 1달러 증가할 때마다,")
print(f"         채무불이행 오즈는 약 {(OR_val-1)*100:.4f}% 증가함")
print(f"   예) balance 1000달러 증가 → OR = exp(0.005499×1000) = {np.exp(beta1_val*1000):.4f}배")


## 문제 2. 변수 선택 — Deviance & AIC

In [ ]:
# ============================================================
# 문제 2. 변수 선택 — Deviance & AIC
# ============================================================
print("=" * 60)
print("📊 문제 2. 변수 선택 — Deviance와 AIC")
print("=" * 60)
print("""
【통계 용어 정리】
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ Deviance (이탈도)
  - 모형이 데이터를 얼마나 잘 설명하지 못하는지 측정
  - 값이 작을수록 모형이 데이터에 잘 맞음
  - Null Deviance: 절편만 있는 모형
  - Residual Deviance: 변수를 추가한 후의 이탈도

▶ AIC (Akaike Information Criterion, 아카이케 정보 기준)
  - AIC = -2 × 로그우도 + 2 × 파라미터 수
  - 모형 복잡도(변수 수)에 패널티를 부과함
  - 값이 작을수록 더 좋은 모형
  - 변수가 많아도 성능이 충분히 좋아야 선택

▶ Deviance 차이 검정 (Likelihood Ratio Test)
  - 두 모형의 Residual Deviance 차이가 통계적으로 유의한지 검정
  - 검정통계량 = Deviance 차이 ~ χ²(자유도 차이)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# Default 데이터에서 더미 변수 준비
Default2 = Default.copy()
Default2['default_num'] = (Default2['default'] == 'Yes').astype(int)
Default2['student_num'] = (Default2['student'] == 'Yes').astype(int)

# 모형 적합
model1 = smf.glm('default_num ~ balance',
                  data=Default2, family=sm.families.Binomial()).fit()
model2 = smf.glm('default_num ~ balance + income',
                  data=Default2, family=sm.families.Binomial()).fit()
model3 = smf.glm('default_num ~ balance + income + student_num',
                  data=Default2, family=sm.families.Binomial()).fit()

print("(1) 각 모형의 Residual Deviance, df, AIC")
print("-" * 55)
print(f"{'모형':<10} {'Residual Dev':>15} {'df':>6} {'AIC':>12}")
print("-" * 55)
for name, m in [("모형 1", model1), ("모형 2", model2), ("모형 3", model3)]:
    print(f"{name:<10} {m.deviance:>15.4f} {m.df_resid:>6.0f} {m.aic:>12.4f}")
print("-" * 55)
print()

print("(2) Deviance 차이 검정 (Likelihood Ratio Test)")
print("-" * 55)

def lr_test(model_small, model_large, label):
    dev_diff = model_small.deviance - model_large.deviance
    df_diff  = model_small.df_resid - model_large.df_resid
    p_value  = stats.chi2.sf(dev_diff, df_diff)
    print(f"  {label}")
    print(f"  Δ Deviance = {dev_diff:.4f}, Δdf = {df_diff}")
    print(f"  p-value = {p_value:.6f}  {'✅ 유의함 (p<0.05)' if p_value < 0.05 else '❌ 유의하지 않음'}")
    print()

lr_test(model1, model2, "모형1 vs 모형2 (income 추가 효과)")
lr_test(model2, model3, "모형2 vs 모형3 (student 추가 효과)")

print("(3) 최적 모형 선택")
print("-" * 55)
print("  AIC 기준: 모형3 AIC가 가장 낮으나 모형2와 차이 미미")
print()
print("  각 모형 오즈비:")
for name, m in [("모형 1", model1), ("모형 2", model2), ("모형 3", model3)]:
    print(f"  [{name}]")
    print(f"  {np.exp(m.params).round(4).to_string()}")
    print()
print("  ✅ 결론:")
print("  - 통계적 유의성: 모형3이 student 추가로 Deviance 개선 (p<0.05)")
print("  - 실질적 유의성: student의 오즈비 효과가 balance에 비해 작음")
print("  - 최적 모형: 모형3 선택 (but 해석 복잡도 vs 성능 개선 트레이드오프 고려)")


## 문제 3. 최종 모형 해석

In [ ]:
# ============================================================
# 문제 3. 최종 모형 해석 (모형3 기준)
# ============================================================
print("=" * 60)
print("🔍 문제 3. 최종 모형 해석 (모형3: balance + income + student)")
print("=" * 60)
print("""
【통계 용어 정리】
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ 유의수준 (Significance Level)
  - 보통 α = 0.05 (5%)를 기준으로 사용
  - p-value < 0.05이면 통계적으로 유의하다고 판단

▶ 회귀계수의 방향
  - 양(+): 변수 증가 → 채무불이행 확률 증가
  - 음(-): 변수 증가 → 채무불이행 확률 감소

▶ logLik (Log-Likelihood, 로그 우도)
  - 모형이 데이터를 잘 설명하는 정도
  - 값이 클수록(0에 가까울수록) 좋음

▶ AIC 공식: AIC = -2 × logLik + 2 × p (p = 파라미터 수)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

final_model = model3
print("최종 모형 summary():")
print(final_model.summary())
print()

print("(1) 유의한 변수 및 방향:")
print("-" * 55)
params = final_model.params
pvalues = final_model.pvalues
for var in params.index:
    sig = "✅ 유의" if pvalues[var] < 0.05 else "❌ 비유의"
    direction = "양(+)" if params[var] > 0 else "음(-)"
    print(f"  {var:<15}: β={params[var]:>10.6f}, p={pvalues[var]:.4e}, {sig}, 방향={direction}")
print()
print("  해석:")
print("  - balance   (+): 잔액이 클수록 채무불이행 확률 증가 ← 직관적")
print("  - income    (-): 소득이 높을수록 채무불이행 확률 감소 ← 직관적")
print("  - student   (-): 학생이면 채무불이행 확률 감소 ← 의외! (balance 통제 후)")
print()

print("(2) 각 변수 오즈비:")
print("-" * 55)
OR = np.exp(params)
CI = np.exp(final_model.conf_int())
for var in params.index:
    if var == 'Intercept': continue
    print(f"  {var:<15}: OR = {OR[var]:.6f}")
    print(f"               95% CI = ({CI.loc[var,0]:.6f}, {CI.loc[var,1]:.6f})")
    if var == 'balance':
        print(f"               → balance 1달러 증가 시 오즈 {(OR[var]-1)*100:.4f}% 증가")
    elif var == 'income':
        print(f"               → income 1달러 증가 시 오즈 {(1-OR[var])*100:.6f}% 감소")
    elif var == 'student_num':
        print(f"               → 학생이면 비학생 대비 오즈 {OR[var]:.4f}배")
    print()

print("(3) logLik와 AIC 직접 계산:")
print("-" * 55)
loglik = final_model.llf
p_params = len(final_model.params)  # 파라미터 수
AIC_manual = -2 * loglik + 2 * p_params
print(f"  logLik (모형 출력값)     = {loglik:.6f}")
print(f"  파라미터 수 (p)          = {p_params}")
print(f"  AIC (직접 계산)         = -2 × {loglik:.4f} + 2 × {p_params} = {AIC_manual:.4f}")
print(f"  AIC (모형 출력값)        = {final_model.aic:.4f}")
print(f"  차이                     = {abs(AIC_manual - final_model.aic):.8f}  ✅ 일치")


## 문제 4. Performance Metrics

In [ ]:
# ============================================================
# 문제 4. Performance Metrics (threshold = 0.5)
# ============================================================
print("=" * 60)
print("📈 문제 4. Performance Metrics (threshold = 0.5)")
print("=" * 60)
print("""
【통계 용어 정리 - Confusion Matrix】
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
               실제 Positive  실제 Negative
예측 Positive |    TP(진양성)  |   FP(위양성)  |
예측 Negative |    FN(위음성)  |   TN(진음성)  |

  TP: 실제 불이행자를 불이행으로 올바르게 예측
  FP: 실제 정상인을 불이행으로 잘못 예측 (False Alarm)
  TN: 실제 정상인을 정상으로 올바르게 예측
  FN: 실제 불이행자를 정상으로 잘못 예측 ← 금융에서 치명적!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# 예측 확률 계산
y_true = Default2['default_num'].values
y_prob = final_model.predict(Default2).values

# threshold = 0.5 적용
threshold = 0.5
y_pred = (y_prob >= threshold).astype(int)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
TN, FP, FN, TP = cm.ravel()
n = len(y_true)

print(f"(1) Confusion Matrix (threshold = {threshold})")
print("-" * 45)
print(f"          예측 정상    예측 불이행")
print(f"실제 정상   TN={TN:5d}    FP={FP:5d}")
print(f"실제불이행  FN={FN:5d}    TP={TP:5d}")
print()

# 지표 계산
Accuracy    = (TP + TN) / n
Sensitivity = TP / (TP + FN)   # Recall
Specificity = TN / (TN + FP)
Precision   = TP / (TP + FP)
F1          = 2 * Precision * Sensitivity / (Precision + Sensitivity)

# Kappa
Pe = ((TP+FP)/n * (TP+FN)/n) + ((TN+FN)/n * (TN+FP)/n)
Kappa = (Accuracy - Pe) / (1 - Pe)

# McNemar
McNemar_stat = (FP - FN)**2 / (FP + FN)
McNemar_pval = stats.chi2.sf(McNemar_stat, df=1)

print("Performance Metrics 표:")
print("-" * 75)
print(f"{'Metric':<15} {'공식':<30} {'계산값':>10}  해석")
print("-" * 75)
metrics = [
    ("Accuracy",    "(TP+TN)/n",                    Accuracy,    "전체 정확도"),
    ("Sensitivity", "TP/(TP+FN)",                   Sensitivity, "실제 불이행 탐지율 (Recall)"),
    ("Specificity", "TN/(TN+FP)",                   Specificity, "실제 정상 탐지율"),
    ("Precision",   "TP/(TP+FP)",                   Precision,   "불이행 예측 중 실제 불이행 비율"),
    ("F1",          "2×Prec×Sens/(Prec+Sens)",       F1,          "Precision-Recall 조화평균"),
    ("Kappa",       "(Acc−Pe)/(1−Pe)",               Kappa,       "우연 보정 일치도"),
    ("McNemar p",   "(FP−FN)²/(FP+FN) → χ²",        McNemar_pval,"FP vs FN 비대칭 검정"),
]
for name, formula, val, interp in metrics:
    print(f"{name:<15} {formula:<30} {val:>10.4f}  {interp}")
print("-" * 75)
print()

print(f"(2) McNemar 검정 분석:")
print(f"   FP = {FP:,}  (정상을 불이행으로 잘못 예측)")
print(f"   FN = {FN:,}  (불이행을 정상으로 잘못 예측)")
print(f"   → {'FP > FN' if FP > FN else 'FN > FP'}: {'정상인을 더 많이 불이행으로 오분류' if FP > FN else '불이행자를 더 많이 정상으로 오분류'}")
print(f"   McNemar p = {McNemar_pval:.4e}  → FP와 FN 비율이 {'유의하게 비대칭' if McNemar_pval<0.05 else '대칭적'}")
print()
print(f"   ✅ 금융 데이터에서: FN이 더 치명적!")
print(f"   FN = 실제 채무불이행자를 정상으로 예측 → 대출 승인 → 손실 발생")
print(f"   FP = 정상 고객을 불이행으로 예측 → 대출 거절 → 기회비용만 발생")
print()

print(f"(3) Kappa 해석:")
print(f"   Kappa = {Kappa:.4f}")
print(f"   해석 기준: 0.6~0.8 = 상당한 일치, 0.4~0.6 = 보통 일치")
kappa_level = "상당한 일치" if Kappa >= 0.6 else "보통 일치" if Kappa >= 0.4 else "낮은 일치"
print(f"   → {kappa_level}")
print()
print(f"   Accuracy만으로 판단하면 안 되는 이유:")
print(f"   - 이 데이터의 불이행 비율 = {y_true.mean()*100:.1f}% (극심한 불균형)")
print(f"   - 아무것도 안 하고 '전부 정상'으로 예측해도 Accuracy = {(1-y_true.mean())*100:.1f}%")
print(f"   - Kappa는 우연에 의한 정확도를 제거하여 실제 모형 성능을 평가")


## 문제 5. ROC / PR Curve 및 최적 Threshold

In [ ]:
# ============================================================
# 문제 5. ROC / PR Curve와 최적 threshold
# ============================================================
print("=" * 60)
print("📉 문제 5. ROC / PR Curve와 최적 threshold")
print("=" * 60)
print("""
【통계 용어 정리】
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ ROC Curve (Receiver Operating Characteristic)
  - X축: FPR = FP/(FP+TN) = 1 - Specificity
  - Y축: TPR = TP/(TP+FN) = Sensitivity (Recall)
  - threshold를 0~1로 움직이면서 그린 곡선

▶ AUC (Area Under Curve)
  - ROC 곡선 아래 면적 (0.5~1.0)
  - 0.5 = 무작위 예측, 1.0 = 완벽한 예측
  - AUC > 0.9 = 우수, 0.8~0.9 = 양호, 0.7~0.8 = 보통

▶ PR Curve (Precision-Recall Curve)
  - X축: Recall (Sensitivity)
  - Y축: Precision
  - 불균형 데이터에서 ROC보다 더 정보량이 많음

▶ Youden Index
  - J = Sensitivity + Specificity - 1
  - 이 값이 최대인 threshold = 최적 임계값
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── (1) ROC Curve ──────────────────────────────────────────
fpr, tpr, thresholds_roc = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

ax = axes[0]
ax.plot(fpr, tpr, color='steelblue', lw=2,
        label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='무작위 예측 (AUC=0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlabel('FPR (1 - Specificity)', fontsize=11)
ax.set_ylabel('TPR (Sensitivity)', fontsize=11)
ax.set_title('(1) ROC Curve', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

print(f"(1) ROC Curve AUC = {roc_auc:.4f}")
print(f"   해석: 임의 양성과 임의 음성을 비교할 때,")
print(f"         양성이 더 높은 확률을 받을 가능성이 {roc_auc*100:.2f}%")
print()

# ── (2) PR Curve ───────────────────────────────────────────
precision_arr, recall_arr, thresholds_pr = precision_recall_curve(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)

ax = axes[1]
ax.plot(recall_arr, precision_arr, color='darkorange', lw=2,
        label=f'PR Curve (AUC = {pr_auc:.4f})')
baseline = y_true.mean()
ax.axhline(y=baseline, color='k', linestyle='--', lw=1,
           label=f'기준선 (불이행 비율={baseline:.3f})')
ax.fill_between(recall_arr, precision_arr, alpha=0.1, color='darkorange')
ax.set_xlabel('Recall (Sensitivity)', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('(2) PR Curve', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

print(f"(2) PR Curve AUC = {pr_auc:.4f}")
print(f"   ROC AUC({roc_auc:.4f})과 비교:")
print(f"   - ROC AUC가 높아도 PR AUC는 낮을 수 있음 (불균형 데이터 특성)")
print(f"   - 불이행 비율({baseline*100:.1f}%)이 낮아 PR Curve가 더 엄격한 기준")
print()

# ── (3) Youden Index 최적 threshold ────────────────────────
youden = tpr - fpr  # Sensitivity + Specificity - 1 = TPR - FPR
best_idx = np.argmax(youden)
best_threshold = thresholds_roc[best_idx]
best_tpr = tpr[best_idx]
best_fpr = fpr[best_idx]

axes[0].scatter(best_fpr, best_tpr, color='red', s=100, zorder=5,
                label=f'최적 threshold={best_threshold:.3f}')
axes[0].legend(loc='lower right', fontsize=9)

print(f"(3) Youden Index 최적 threshold:")
print(f"   최적 threshold = {best_threshold:.4f}")
print(f"   해당 Sensitivity = {best_tpr:.4f}")
print(f"   해당 Specificity = {1-best_fpr:.4f}")
print(f"   Youden Index = {youden[best_idx]:.4f}")
print()

# ── (4) threshold 비교 ─────────────────────────────────────
y_pred_05  = (y_prob >= 0.5).astype(int)
y_pred_opt = (y_prob >= best_threshold).astype(int)

def get_metrics(y_t, y_p):
    cm_ = confusion_matrix(y_t, y_p)
    tn_, fp_, fn_, tp_ = cm_.ravel()
    sens = tp_/(tp_+fn_)
    spec = tn_/(tn_+fp_)
    return sens, spec

sens_05,  spec_05  = get_metrics(y_true, y_pred_05)
sens_opt, spec_opt = get_metrics(y_true, y_pred_opt)

print(f"(4) threshold 비교:")
print(f"{'':>20} {'threshold=0.5':>15} {'최적 threshold':>15}")
print(f"{'threshold':>20} {0.5:>15.4f} {best_threshold:>15.4f}")
print(f"{'Sensitivity':>20} {sens_05:>15.4f} {sens_opt:>15.4f}")
print(f"{'Specificity':>20} {spec_05:>15.4f} {spec_opt:>15.4f}")
print()

# ── (5) 금융 데이터 threshold 선택 ─────────────────────────
print(f"(5) 금융 데이터(채무불이행 탐지)에서 threshold 선택:")
print(f"   → 최적 threshold ({best_threshold:.3f}) 선택 권장")
print()
print(f"   이유:")
print(f"   1) FN(불이행자를 정상으로 오분류)이 더 치명적인 손실을 초래")
print(f"   2) threshold를 낮추면 Sensitivity 증가 → FN 감소")
print(f"   3) threshold 0.5: Sensitivity={sens_05:.4f} → 불이행자 {sens_05*100:.1f}% 탐지")
print(f"   4) 최적 threshold: Sensitivity={sens_opt:.4f} → 불이행자 {sens_opt*100:.1f}% 탐지")
print(f"   5) 실무에서는 비용-편익 분석으로 threshold를 더 세밀하게 조정")

# ── Threshold 비교 시각화 ───────────────────────────────────
ax = axes[2]
labels = [f'threshold=0.5\n(Sens={sens_05:.3f}, Spec={spec_05:.3f})',
          f'최적 threshold={best_threshold:.3f}\n(Sens={sens_opt:.3f}, Spec={spec_opt:.3f})']
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [sens_05, sens_opt], width, label='Sensitivity', color='steelblue', alpha=0.8)
ax.bar(x + width/2, [spec_05, spec_opt], width, label='Specificity', color='darkorange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f'threshold=0.5', f'최적({best_threshold:.3f})'], fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('값', fontsize=11)
ax.set_title('(4) Threshold 비교', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for i, (s, sp) in enumerate([(sens_05, spec_05), (sens_opt, spec_opt)]):
    ax.text(i - width/2, s + 0.02, f'{s:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, sp + 0.02, f'{sp:.3f}', ha='center', fontsize=9)

plt.suptitle('문제 5: ROC / PR Curve 및 최적 Threshold 분석',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('problem5_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ 그래프 저장 완료: problem5_curves.png")
